# Algorithm 4 — Acceptance-Weighted Throughput

**File:** `src/CopilotScope.Collector/Quality/Analyzers.cs` (lines 49–90)

Measures the **productivity density** of an AI session: how much accepted code is produced per unit
of interaction, discounted by rejections.

---

## 1  Component Metrics

Let $a$ = edits accepted, $j$ = edits rejected, $L$ = lines added, $T$ = number of turns,
$O$ = output tokens.

$$
\alpha = \frac{a}{a + j} \qquad \text{(acceptance ratio)}
$$

$$
\ell_{\text{adj}} = \frac{L \cdot \alpha}{T} \qquad \text{(accepted LOC per turn)}
$$

$$
\ell_{\text{tok}} = \frac{L}{O / 1000} \qquad \text{(LOC per 1k output tokens, optional)}
$$

---

## 2  Throughput Score

$$
\boxed{\text{Score} = 0.6 \cdot \alpha + 0.4 \cdot \min\!\left(1,\; \frac{\ell_{\text{adj}}}{20}\right)}
$$

The LOC component **saturates at 20 accepted LOC/turn**.  Above that threshold the model's
utility is capped because throughput beyond a certain rate can indicate auto-accepting without
review.  Acceptance ratio dominates (0.6 weight) because a high LOC rate with many rejections
is noise, not productivity.

---

## 3  Finding Thresholds

| Condition | Finding |
|---|---|
| $\alpha < 0.5$ and $a+j \geq 4$ | Suggestions miss the mark — more rejections than acceptances |
| $\ell_{\text{adj}} \geq 10$ | High effective throughput |
| Otherwise | Moderate throughput |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def throughput(accepted, rejected, lines_added, turns, output_tokens=0):
    edits = accepted + rejected
    alpha = accepted / edits if edits > 0 else 1.0
    T = max(1, turns)
    loc_per_turn = lines_added / T
    adjusted = lines_added * alpha / T
    out_k = output_tokens / 1000.0
    loc_per_ktok = lines_added / out_k if out_k > 0 else None
    score = np.clip(0.6 * alpha + 0.4 * min(1.0, adjusted / 20.0), 0, 1)

    if alpha < 0.5 and edits >= 4:
        finding = 'Suggestions miss the mark'
    elif adjusted >= 10:
        finding = 'High effective throughput'
    else:
        finding = 'Moderate throughput'

    return {'alpha': alpha, 'loc_per_turn': loc_per_turn, 'adjusted': adjusted,
            'loc_per_ktok': loc_per_ktok, 'score': score, 'finding': finding}

cases = [
    ('High acceptance, high LOC',   40, 5,  200, 10, 15000),
    ('Low acceptance rate',         10, 15,  80, 8,  12000),
    ('Good acceptance, low LOC',    20, 4,   30, 10,  8000),
    ('Perfect acceptance, moderate', 30, 0, 120, 15, 20000),
    ('No edit telemetry',            0, 0,  50, 5,   5000),
]

print(f'{"Scenario":<35} {"alpha":>6} {"adj_loc/t":>10} {"Score":>7}  Finding')
print('-' * 90)
for name, a, j, L, T, tok in cases:
    r = throughput(a, j, L, T, tok)
    print(f'{name:<35} {r["alpha"]:>6.2f} {r["adjusted"]:>10.1f} {r["score"]:>7.3f}  {r["finding"]}')

In [ ]:
# Score surface: acceptance ratio vs. adjusted LOC per turn
alpha_range = np.linspace(0, 1, 200)
loc_range   = np.linspace(0, 30, 200)
A, L2 = np.meshgrid(alpha_range, loc_range)
S = 0.6 * A + 0.4 * np.minimum(1.0, L2 / 20.0)

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(A, L2, S, levels=20, cmap='YlGn')
ct = ax.contour(A, L2, S, levels=[0.5, 0.7, 0.9], colors='black', linewidths=1)
ax.clabel(ct, fmt='%.1f', fontsize=9)
plt.colorbar(cs, ax=ax, label='Throughput score')
ax.axvline(0.5, ls='--', color='red', alpha=0.6, label='alpha=0.5 threshold')
ax.axhline(10, ls='--', color='blue', alpha=0.6, label='high throughput at 10 LOC/turn')
ax.set_xlabel('Acceptance ratio $\\alpha$')
ax.set_ylabel('Adjusted LOC per turn $\\ell_{adj}$')
ax.set_title('Throughput score surface')
ax.legend()
plt.tight_layout()
plt.savefig('throughput_surface.png', dpi=150)
plt.show()